# 2. Análisis exploratorio de la tabla curada
**Entrada:** una versión real de `model_table.csv` y su reporte. Sustituya `VERSION`.
El EDA describe faltantes, distribuciones, constantes, relaciones y posibles anomalías.
No demuestra capacidad predictiva. Decisiones estadísticas de selección o transformación
se vuelven a estimar dentro de entrenamiento; no utilizar el test para escoger variables.


In [ ]:
from pathlib import Path

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'src' / 'ingesta.py').is_file() and (p / 'config' / 'data_schema.yml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Abra Jupyter desde la raíz del repositorio o notebooks/.')

import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt

DATA = ROOT / 'data' / 'processed' / 'VERSION' / 'model_table.csv'
df = None
if not DATA.is_file():
    print('Indique una versión real generada por src/preprocesamiento.py; no hay EDA que ejecutar.')
else:
    report = json.loads((DATA.parent / 'data_quality_report.json').read_text(encoding='utf-8'))
    if hashlib.sha256(DATA.read_bytes()).hexdigest() != report['output_sha256']:
        raise ValueError('La tabla cambió respecto de su reporte de curaduría.')
    roles = report['roles']
    df = pd.read_csv(
        DATA,
        dtype={c: 'string' for c in roles['identifiers'] + roles['groups']},
        keep_default_na=False, na_values=[''],
    )
    numeric_columns = roles['predictors'] + [roles['target']]
    df.head()


In [ ]:
if df is not None:
    df.info()
    df.describe(include='all').T


In [ ]:
if df is not None:
    df.isna().sum().sort_values(ascending=False)


In [ ]:
if df is not None:
    df[numeric_columns].hist(figsize=(12, 8))
    plt.tight_layout()
    plt.show()


In [ ]:
if df is not None:
    # Relaciones exploratorias; no seleccionan predictores automáticamente.
    correlation = df[numeric_columns].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(8, 6))
    img = ax.imshow(correlation, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(correlation)), correlation.columns, rotation=90)
    ax.set_yticks(range(len(correlation)), correlation.index)
    fig.colorbar(img, ax=ax, label='Correlación de Spearman')
    plt.tight_layout()
    plt.show()


## Revisión antes del modelamiento
Verificar unidades, target, exclusiones, duplicados, constantes, redundancia y anomalías físicas
contra el reporte y la fuente. Confirmar réplicas y grupos antes de diseñar las particiones.
Los identificadores, grupos y `__source_record` son trazabilidad, nunca predictores.


## Entrega de esta etapa
Documentar distribución del target, patrones de ausencia, posibles réplicas y representaciones
geométricas alternativas. Una correlación entre volumen y dimensiones puede deberse a su fórmula,
no a nueva información. Continuar con `03_Particiones_y_preprocesamiento.ipynb`.
